# Árvore de Decisão

Este bloco de código é o ponto de partida. Primeiro, ele **importa** todas as bibliotecas que serão usadas no projeto:

**pandas:** É a principal ferramenta para manipulação de dados em Python. Usamos para ler o arquivo csv e organizá-lo em uma tabela (DataFrame).

**sklearn:** É a biblioteca de Machine Learning. Dela, usamos o LabelEncoder e o OneHotEncoder para transformar os textos em números, e o DecisionTreeClassifier para criar nosso modelo.

**matplotlib e tree:** São usados para visualizar a árvore de decisão no final do processo.

A linha base = pd.read_csv(...) lê o seu arquivo e armazena os dados na variável base, que será usada em todas as etapas seguintes.

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree
import matplotlib.pyplot as plt

# 1. Carregar os dados
try:
    base = pd.read_csv('/content/sample_data/restaurante.csv', sep=';')
except FileNotFoundError:
    print("Arquivo 'restaurante.csv' não encontrado. Certifique-se de que ele foi carregado.")

Esta é a etapa **mais importante** da preparação.

Primeiro, seguimos a regra da sua atividade, aplicando uma **codificação manual** na coluna Cliente. Usamos um dicionário (map) para garantir que "Nenhum" seja 0, "Alguns" seja 1 e "Cheio" seja 2.

Depois, para as outras colunas de texto que são binárias (Sim/Não) ou ordinais, usamos o **LabelEncoder**. Ele transforma o texto em números de forma automática.

Por fim, para a coluna Tipo, que possui várias categorias sem uma ordem lógica entre elas (Frânces, Italiano, etc.), usamos o **OneHotEncoder**. Isso evita que o modelo aprenda uma relação de ordem que não existe (por exemplo, que 2 > 1). Ele cria colunas de "verdadeiro/falso" para cada tipo de restaurante.

In [ ]:
# 2. Tratamento dos dados
# Codificação personalizada para 'Cliente'
cliente_map = {'Nenhum': 0, 'Alguns': 1, 'Cheio': 2}
base['Cliente'] = base['Cliente'].map(cliente_map)

# LabelEncoder para outras colunas ordinais
cols_label_encode = ['Alternativo', 'Bar', 'SexSab','fome', 'Cliente', 'Preco', 'Chuva', 'Res','Tempo']
base[cols_label_encode] = base[cols_label_encode].apply(LabelEncoder().fit_transform)

# OneHotEncoder para a coluna 'Tipo'
onehot_encoder = OneHotEncoder(sparse_output=False)
tipo_encoded = onehot_encoder.fit_transform(base[['Tipo']])
tipo_df = pd.DataFrame(tipo_encoded, columns=onehot_encoder.get_feature_names_out(['Tipo']))
base = pd.concat([base.drop('Tipo', axis=1), tipo_df], axis=1)

Por convenção, chamamos o **conjunto de atributos** de **X** e a **coluna alvo** de **y**. X contém todas as informações que damos ao modelo para ele tomar uma decisão (se tem bar, se o cliente tem fome, etc.), e y é a decisão final que ele precisa aprender a prever ("Sim" ou "Não").

In [ ]:
# 3. Separação das variáveis
X = base.drop('Conclusao', axis=1)
y = base['Conclusao']

Aqui, a **"mágica"** acontece. A linha modelo.fit(X, y) pega os atributos X e as respostas y e executa o algoritmo da **árvore de decisão**. Ele analisa todos os dados e descobre as melhores "perguntas" (nós) e em que ordem fazê-las para chegar a uma previsão sobre a Conclusao.

In [ ]:
# 4. Treinamento do modelo de Árvore de Decisão
modelo = DecisionTreeClassifier(criterion='entropy', random_state=42)
modelo.fit(X, y)

Este bloco final serve para visualizar o resultado do treinamento. A função plot_tree é altamente customizável:

feature_names: Garante que, em vez de vermos "Coluna 1 <= 0.5", vejamos "fome <= 0.5", o que torna a árvore legível.

class_names: Garante que as folhas da árvore (as respostas finais) mostrem "Sim" ou "Não".

filled e rounded: Apenas melhoram a estética e a clareza da visualização.

savefig: Permite que você salve o resultado como uma imagem para usar em seu trabalho ou apresentação.

In [ ]:
# 5. Geração e salvamento da imagem da árvore
fig, axes = plt.subplots(nrows=1, ncols=1, figsize=(15, 15))
tree.plot_tree(modelo, feature_names=X.columns.tolist(), class_names=sorted(y.unique()), filled=True, rounded=True)
plt.title("Árvore de Decisão para a Base 'Restaurante' com Codificação Personalizada")
fig.savefig("decision_tree_restaurante.png")

print("A imagem da árvore de decisão foi salva como 'decision_tree_restaurante.png'")